In [4]:
import sys
import os
import ast
import pandas as pd
import numpy as np # NaN 체크용

# 1. 경로 설정
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_loader import load_config

# 2. 데이터 로드
cfg = load_config()
real_data_path = os.path.join(project_root, cfg.path.data.test)
df = pd.read_csv(real_data_path)

# ---------------------------------------------------------
# 🛠️ [핵심] 안전한 파싱 함수 (Fallback Logic)
# ---------------------------------------------------------
def get_problem_data(row):
    """
    1순위: 'question', 'choices' 컬럼에서 가져오기
    2순위: 실패 시 'problems' 컬럼(딕셔너리 스트링) 파싱해서 가져오기
    """
    q = row.get('question')
    c = row.get('choices')

    # 1. 개별 컬럼에 데이터가 잘 들어있는 경우
    if pd.notna(q) and q != "" and pd.notna(c) and c != "":
        # choices가 문자열이면 리스트로 변환
        if isinstance(c, str):
            try:
                c = ast.literal_eval(c)
            except:
                c = []
        return q, c

    # 2. 개별 컬럼이 비어있다면 'problems' 컬럼 파싱 (비상 대책)
    try:
        if pd.notna(row.get('problems')):
            problems_dict = ast.literal_eval(row['problems'])
            return problems_dict.get('question', ''), problems_dict.get('choices', [])
    except Exception as e:
        print(f"⚠️ 파싱 실패: {e}")
    
    return "질문 없음", []

# ---------------------------------------------------------
# 3. State 생성
# ---------------------------------------------------------
row = df.iloc[654]
question, choices = get_problem_data(row)

dummy_state = {
    # --- 입력 데이터 ---
    "paragraph": row.get('paragraph', ''),
    "problem": {
        "question": question,
        "choices": choices
    },

    # --- 처리 데이터 초기화 ---
    "track_info": {},
    "retrieved_context": [],
    "final_prompt_messages": [],
    "solver_results": [],
    "final_answer": None
}

# ---------------------------------------------------------
# 4. 디버깅 출력
# ---------------------------------------------------------
print("\n=== 🔍 데이터 확인 ===")
print(f"1. 컬럼 목록: {df.columns.tolist()}") # 혹시 공백이 있는지 확인용
print(f"2. 원본 problems 데이터: {row.get('problems')}") # 여기가 비어있는지 확인
print("-" * 30)
print("=== ✅ AgentState 구조 확인 ===")
print(f"🔹 지문: {dummy_state['paragraph']}")
print(f"🔹 질문: {dummy_state['problem']['question']}")
print(f"🔹 선택지: {dummy_state['problem']['choices']}")
print(f"🔹 선택지 갯수: {len(dummy_state['problem']['choices'])}")


=== 🔍 데이터 확인 ===
1. 컬럼 목록: ['Unnamed: 0', 'id', 'paragraph', 'problems', 'question_plus']
2. 원본 problems 데이터: {'question': '다음 중 위 작품이 대영제국의 영향력 또는 지배 하에 있는 원주민들에 대해 취하고 있는 태도는?', 'choices': ['영국인의 “추방된 자식”들이다.', '“야만인”이지만 지속되는 평화를 위해 전쟁을 했다.', '기독교로 개종될 준비가 되어 있다.', '문명화와 발달이 더딘 문명의 산물이다.'], 'answer': ''}
------------------------------
=== ✅ AgentState 구조 확인 ===
🔹 지문: 백인의 짐을 떠맡아라—
최고의 품종을 보내어라—
가서 네 아들들을 묶어 유배시켜
포로들의 필요를 충족하여라
무거운 밧줄에 묶여 기다려라
설렌 사람들과 야만인—
그대의 새로 붙잡힌 음침한 민족들
반은 악마이고 반은 아이인 자들 위에서.
...
백인의 짐을 떠맡아라—
잔혹한 평화의 전쟁—
기근의 입을 가득 채우라
질병이 멈추도록 명하라
목표가 가장 가까워올 때
다른 이들이 추구하는 종말이란…
나태와 이방인의 어리석음을 경계하라
모든 희망을 없애노라
러디야드 키플링, 백인의 짐, 1899년
🔹 질문: 다음 중 위 작품이 대영제국의 영향력 또는 지배 하에 있는 원주민들에 대해 취하고 있는 태도는?
🔹 선택지: ['영국인의 “추방된 자식”들이다.', '“야만인”이지만 지속되는 평화를 위해 전쟁을 했다.', '기독교로 개종될 준비가 되어 있다.', '문명화와 발달이 더딘 문명의 산물이다.']
🔹 선택지 갯수: 4


In [5]:
from src.agent.nodes.retrieval import RetrievalNode

retriever = RetrievalNode(cfg)
print(f"▶️ 검색 시작: {dummy_state['problem']['question']}")

result = retriever(dummy_state)
print(f"DEBUG: 결과 딕셔너리 키 목록: {result.keys()}")
print(f"DEBUG: 검색된 문서 수: {len(result.get('retrieved_context', []))}")

if result.get("retrieved_context"):
    for i, txt in enumerate(result["retrieved_context"]):
        print(f"[{i}] {txt[:250]}...")
else:
    print("❌ 검색 결과가 비어 있습니다.")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

▶️ 검색 시작: 다음 중 위 작품이 대영제국의 영향력 또는 지배 하에 있는 원주민들에 대해 취하고 있는 태도는?


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


DEBUG: 결과 딕셔너리 키 목록: dict_keys(['retrieved_context'])
DEBUG: 검색된 문서 수: 5
[0] 문서 제목: 백인의 짐

섬네일
〈백인의 짐〉 (The White Man's Burden)은 영국 작가 러디어드 키플링이 1899년 2월에 발표한 시이며, 이 시가 내세우는 서구의 제국주의 이데올로기이다. 키플링은 미개한 인종을 올바르게 이끄는 것이 백인이 져야할 짐, 백인의 의무라고 역설한다.
1899년 2월, 에스파냐가 물러난 필리핀을 미국이 침략한다. (필리핀-미국 전쟁)
키플링은 이에 호응하여 〈백인의 짐—미국과 필리핀 제도〉를 발표한다...
[1] 문서 제목: 아프리카 분할

. 식민지화 국가의 일부 사람들은 조지프 콘래드의 《암흑의 심연》(1899)(키플링의 《백인의 짐》과 비슷한 시기에 출판됨)이나 루이페르디낭 셀린의 《밤 끝으로의 여행》 (1932)에 묘사된 것처럼, 식민지 행정이 스스로 방치될 때 발생하는 불필요한 악에 반대했다...
[2] 문서 제목: 이스라엘-팔레스타인 관계

.
근대적인 의미에서 이스라엘과 팔레스타인의 존재는, 제1차 세계 대전 당시 대영 제국과 적대국인 오스만 제국 지배 하에 있었던 팔레스타인 지방에 팔레스타인인과 유대인의 국가 건국에 대한 외교적 약속인 밸푸어 선언과 후세인-맥마흔 서한에 의해서다. Decolonize Palestine|날짜=2021-03-17|언어=en-US|확인날짜=2025-04-09}} 오스만 제국이 전쟁에서 패배하며 이곳은 영국의 위...
[3] 문서 제목: 투스카로라 전쟁

투스카로라 전쟁()은 1711년 가을부터 1715년 2월 11일까지, 당시 영국의 식민지였던 노스캐롤라이나에서 원주민인 투스카로라 족 인디언의 영토를 둘러싸고, 영국, 네덜란드 및 독일 정착민이 투스카로라 족에 행했던 인종 청소(‘미국 인디언 전쟁’)였다. 평화 조약은 1715년에 체결됐다.
배경.
유럽 백인의 노스캐롤라이나의 첫 영구 식민지 이주는 1653년에 본격적으로 시작되었다. 미국